# Computer vision · Comprendre et préparer une image

Une photo de rue contient un bus, des personnes, des façades et beaucoup de détails.
Nous voulons qu'un programme en tire une information simple : **repérer les bus visibles
et produire un compte-rendu que l'on peut vérifier**. Il faudra d'abord comprendre
ce que ce programme reçoit réellement : des nombres, pas des objets déjà nommés.

Nous suivrons les mêmes images tout au long du parcours : les lire avec **OpenCV**,
leur associer une catégorie avec un modèle de **classification**, puis localiser
plusieurs objets avec un **détecteur**. Nous utiliserons **YOLO**, une famille de
modèles, dans deux variantes : l’une classe, l’autre détecte. Enfin, nous réunirons
ces gestes dans une petite application. Les deux modèles sont déjà entraînés :
nous allons les utiliser et examiner leurs erreurs, pas les réentraîner.

![La photo est lue, préparée avec OpenCV, analysée par un modèle puis interprétée par une règle explicite.](../ressources/illustrations/01_fil_conducteur.svg)

**Commençons par la question et l'image.** Les premières explications se lisent
sans lancer de code ; la mise en place intervient à la première manipulation.

Les **figures de référence** sont enregistrées dans ce cours : elles restent
visibles sans exécuter les cellules. Le code produit vos propres résultats ;
si vous changez une image ou un réglage, ils peuvent différer de ces références.


## 1. Regarder la scène avant de demander une réponse

<img src="../donnees/images/bus.jpg" alt="Photo de rue : un bus et plusieurs personnes, utilisée dans les exemples officiels Ultralytics." width="400">

La **computer vision**, ou vision par ordinateur, regroupe les méthodes qui extraient
des informations à partir d'images. Ici, le programme ne sait pas spontanément
ce qu'est un bus. Nous devons choisir **la question**, **la méthode** et **la façon
de vérifier la réponse**.

La **classification** associe une catégorie à l'image entière. Une **classe** est
simplement le nom d'une catégorie connue du modèle. La **détection d'objets**
cherche en plus la position de plusieurs objets, chacun entouré d'un rectangle,
appelé **boîte englobante**. Une étiquette globale ne donne ni un nombre d'objets
ni leurs positions.

![La classification donne une étiquette à toute la scène, la détection donne une boîte et une classe à chaque objet repéré.](../ressources/illustrations/01_deux_questions.svg)

Avant le modèle : voyez-vous un seul objet dans cette photo ? Une seule étiquette
suffira-t-elle pour compter les bus ? Gardez cette distinction pendant les manipulations.


## 2. Une image est un tableau de pixels

Un **pixel** est une petite case de l'image. Une image couleur peut être représentée
par trois nombres par case : rouge (**R**), vert (**G**) et bleu (**B**).
Chaque composante est appelée un **canal**. Pour nos images, les valeurs vont
de **0 à 255** : 0 signifie aucune contribution du canal, 255 sa valeur maximale.
Ce sont des intensités numériques, pas des noms de couleur.

![Image RGB de deux lignes et trois colonnes ; le pixel à la ligne zéro et à la colonne deux contient zéro rouge, zéro vert et 255 bleu.](../ressources/illustrations/01_pixels_canaux.svg)

`[0, 0, 255]` est bleu **si l'ordre est RGB**. `[255, 255, 255]` donne du blanc,
`[0, 0, 0]` du noir. La même suite de nombres n'a plus le même sens si l'ordre
des canaux change. Nous retrouverons ce piège avec OpenCV.

La **résolution** décrit le nombre de pixels en largeur et en hauteur.
Plus de pixels peut conserver plus de détails, mais demande aussi plus de mémoire
et de calcul. Agrandir une petite image n'invente pas les détails perdus.


## 3. Retrouver un pixel et découper une zone

L'origine est en haut à gauche. **x** avance vers la droite et **y** vers le bas.
Python numérote les positions à partir de **0**. Attention : un tableau de pixels
s'indexe d'abord par la **ligne**, puis par la **colonne**, donc `image[y, x]`.

![Les coordonnées de dessin sont x puis y ; le tableau s'indexe par ligne puis colonne.](../ressources/illustrations/01_coordonnees.svg)

**Recadrer**, c'est garder seulement une zone : `image[y1:y2, x1:x2]`.
La borne de fin est exclue. Par exemple, `image[0:2, 1:4]` garde deux lignes
et trois colonnes. Un recadrage retire du contexte ; il ne réduit pas
uniformément toute l'image comme un redimensionnement.


## 4. Ouvrir notre première image avec OpenCV

**OpenCV** est une bibliothèque : un ensemble de fonctions disponibles en Python
pour lire et transformer des images. Son nom dans le code est `cv2`.
**NumPy** représente les pixels par des tableaux. **Matplotlib** les affiche
dans le notebook. Ces outils ne sont pas le modèle appris.

Préparer le Python local avec [INSTALLATION.txt](../INSTALLATION.txt), puis choisir
le noyau `.venv` dans VS Code. Un **noyau** est le processus Python qui exécute
les cellules. Exécuter de haut en bas avec `Maj+Entrée` : une cellule suivante
réutilise les variables créées au-dessus.

Le bloc suivant prépare les chemins ; il ne transforme aucune image.
La vision commence à la cellule suivante, avec la lecture de la photo.
`Path.cwd()` désigne le dossier courant, `.parents` ses dossiers parents ;
`next(...)` retient le premier dossier contenant nos ressources.
`import` charge une bibliothèque et `as` lui donne un nom court.


In [ ]:
from pathlib import Path  # Path manipule un chemin sans écrire les séparateurs Windows à la main.
import cv2               # OpenCV : lire et transformer les pixels.
import numpy as np       # NumPy : manipuler les tableaux de nombres.
import matplotlib.pyplot as plt  # Matplotlib : afficher les images dans le notebook.

# Chercher le dossier du cours depuis ce notebook ou depuis la racine ouverte dans VS Code.
candidats = [Path.cwd(), *Path.cwd().parents]  # Dossier courant, puis ses dossiers parents.
ROOT = next(p for p in candidats if (p / "modeles").is_dir() and (p / "donnees").is_dir())
IMAGES = ROOT / "donnees" / "images"  # Le symbole / assemble les morceaux du chemin.
TRAVAUX = ROOT / "travaux"
TRAVAUX.mkdir(exist_ok=True)  # Créer le dossier de sorties s'il n'existe pas encore.
print("Dossier du cours :", ROOT)


In [ ]:
chemin_image = IMAGES / "bus.jpg"  # Le fichier fourni : aucune caméra n'est ouverte.
image_bgr = cv2.imread(str(chemin_image))  # Lire le fichier ; str donne le chemin en texte.
if image_bgr is None:  # OpenCV renvoie None s'il ne réussit pas à lire le fichier.
    raise FileNotFoundError(f"Image illisible : {chemin_image}")

print("Dimensions du tableau :", image_bgr.shape)  # (hauteur, largeur, nombre de canaux)
print("Type des nombres :", image_bgr.dtype)  # uint8 : entiers non signés sur 8 bits.
print("Pixel en haut à gauche, ordre BGR :", image_bgr[0, 0])


`shape` décrit la forme du tableau : **hauteur, largeur, canaux**.
`dtype` décrit la manière de stocker chaque nombre. `uint8` utilise huit bits
sans signe ; ses 256 valeurs possibles vont de 0 à 255. Un **bit** est une
position qui peut valoir 0 ou 1 ; huit bits permettent de représenter 256 valeurs.

**OpenCV lit ici les canaux dans l'ordre BGR**, bleu-vert-rouge.
Matplotlib attend du **RGB** pour afficher une image couleur. La conversion
ci-dessous remet les mêmes canaux dans l'ordre attendu ; elle ne reconnaît aucun objet.


In [ ]:
image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)  # BGR vers rouge-vert-bleu.
plt.figure(figsize=(6, 7))  # Taille de la figure d'affichage, pas taille de l'image en pixels.
plt.imshow(image_rgb)
plt.title("Notre image de départ — couleurs correctement affichées")
plt.axis("off")  # Masquer les graduations pour regarder la scène.
plt.show()


### Le piège des canaux, visible sur trois pixels

Nous fabriquons trois cases **RGB** avec `np.array`, qui construit un tableau
à partir de listes. Une seule ligne contient ici trois pixels. L'exemple est
volontairement minuscule pour relier chaque couleur à ses trois nombres.


In [ ]:
mini_rgb = np.array([[[255, 0, 0], [0, 255, 0], [0, 0, 255]]], dtype=np.uint8)
# Le premier canal passe de rouge à bleu si l'on traite à tort ce tableau RGB comme du BGR.
mauvais_ordre = cv2.cvtColor(mini_rgb, cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 2, figsize=(9, 2))  # Deux images côte à côte.
for axe, image, titre in zip(axes, [mini_rgb, mauvais_ordre], ["RGB : rouge, vert, bleu", "Canaux inversés : bleu, vert, rouge"]):
    axe.imshow(image)
    axe.set_title(titre)
    axe.axis("off")
plt.show()


**Figure de référence · visible sans exécution**

![À gauche un tableau RGB correctement affiché ; à droite ses canaux rouge et bleu ont été échangés à tort.](../ressources/illustrations/01_canaux_rgb_bgr.png)

À gauche, le tableau est déjà en RGB. À droite, une conversion BGR→RGB appliquée
à tort échange rouge et bleu. Il faut connaître l'ordre initial avant de convertir.


## 5. Changer la taille ou le cadrage : deux opérations différentes

**Redimensionner** modifie le nombre de pixels. Nous gardons le rapport
largeur/hauteur pour ne pas étirer le bus. L'**interpolation** est la règle
qui calcule les pixels de la nouvelle taille ; `INTER_AREA` convient ici
à une réduction en combinant des zones de pixels.

**Recadrer** sélectionne une région sans redimensionner les objets restants.
Dans `cv2.resize`, la taille s'écrit **(largeur, hauteur)**, alors que `shape`
affiche **(hauteur, largeur, canaux)**.

Sur notre photo, la largeur passe de **810 à 360 pixels**. Pour garder les proportions,
la hauteur devient `1080 × 360 / 810 = 480` pixels.
Le rectangle orange ci-dessous désigne une autre opération : garder les colonnes
`20` à `799` et les lignes `200` à `749`. Ces bornes choisissent une zone du bus
et des personnes ; elles ne sont pas calculées par un modèle.
`.copy()` rend cette zone indépendante : la modifier ne changera pas l'image d'origine.


In [ ]:
hauteur, largeur = image_bgr.shape[:2]  # :2 garde seulement hauteur et largeur.
petite_largeur = 360
petite_hauteur = round(hauteur * petite_largeur / largeur)
petite_bgr = cv2.resize(image_bgr, (petite_largeur, petite_hauteur), interpolation=cv2.INTER_AREA)
zone_bgr = image_bgr[200:750, 20:800].copy()  # Lignes puis colonnes, fin exclue.

repere_bgr = image_bgr.copy()  # Le rectangle indique d'où vient la zone extraite.
cv2.rectangle(repere_bgr, (20, 200), (799, 749), (0, 140, 255), 7)
images = [repere_bgr, petite_bgr, zone_bgr]
titres = ['Original : 810 × 1080 px\nRectangle = zone à extraire',
          'Réduction : 360 × 480 px\nToute la scène est conservée',
          'Recadrage : 780 × 550 px\nSeul le rectangle est conservé']
fig, axes = plt.subplots(1, 3, figsize=(13, 5))
for axe, image, titre in zip(axes, images, titres):
    axe.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    axe.set_title(titre)
    axe.axis('off')
plt.tight_layout()
plt.show()
print('Image réduite :', petite_bgr.shape, '— Zone extraite :', zone_bgr.shape)


**Figure de référence · visible sans exécution**

![Original avec rectangle de découpe, image réduite entière et zone extraite.](../ressources/illustrations/01_reduire_ou_recadrer.png)

Suivez le rectangle orange : il devient le troisième panneau. Le panneau central,
lui, garde toute la scène.


Les panneaux sont adaptés à la page : leur taille affichée ne mesure pas le
nombre de pixels. Lisez les dimensions dans les titres. Comparez les objets visibles : la réduction conserve le cadrage global ; la zone
extraite en supprime une partie. Dans les deux cas, certaines informations changent.
Un modèle doit recevoir **la préparation prévue pour lui**. Réduire à n'importe
quelle taille ou supprimer des bords au hasard n'améliore pas automatiquement sa réponse.


## 6. Faire ressortir une propriété des pixels, pas le nom d'un objet

Une image en **niveaux de gris** contient une valeur d'intensité par pixel,
calculée à partir des canaux couleur. Elle perd l'information de couleur.
Un **flou gaussien** mélange les pixels voisins pour atténuer les variations
fines. Le **noyau** est ici la petite zone de voisinage utilisée : `(9, 9)`
signifie neuf pixels de large et neuf de haut. Ici, ce mot désigne un voisinage
de pixels, **pas le noyau Python de Jupyter** qui exécute les cellules.

**Puis, poser une question à chaque pixel.** Un **seuillage** sépare les pixels selon une règle simple : au-dessus d'une valeur,
blanc ; sinon, noir. Le résultat est un **masque binaire**, une image à deux valeurs.
Un pixel blanc signifie ici « assez lumineux », **pas « appartient à un bus »**.

Avec la règle **strictement supérieur à 150**, les intensités `149`, `150`, `151`
deviennent respectivement `0`, `0`, `255`. L'égalité reste noire. Vérifions d'abord
ces trois nombres, avant d'appliquer exactement la même règle à toute la photo.


In [ ]:
intensites = np.array([[149, 150, 151]], dtype=np.uint8)  # Trois pixels, de 0 à 255.
_, resultat_seuil = cv2.threshold(intensites, 150, 255, cv2.THRESH_BINARY)
print('Avant :', intensites.tolist(), '→ Après :', resultat_seuil.tolist())
# _ reçoit une valeur que nous n'utilisons pas : le seuil, déjà connu (150).


In [ ]:
gris = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)  # Trois canaux deviennent une intensité.
flou = cv2.GaussianBlur(gris, (9, 9), sigmaX=0)  # 0 : étalement calculé par OpenCV.
_, masque = cv2.threshold(flou, 150, 255, cv2.THRESH_BINARY)
# La même règle > 150 porte ici sur tous les pixels de l'image floutée.
zoom_gris = gris[300:450, 300:500]  # Même zone du bus pour comparer les détails.
zoom_flou = flou[300:450, 300:500]
images = [gris, masque, zoom_gris, zoom_flou]
titres = ['Gris : intensité, sans couleur', 'Après flou puis seuil : pixels > 150',
          'Zoom du bus avant flou', 'Même zoom après flou 9 × 9']
fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for axe, image, titre in zip(axes.flat, images, titres):
    axe.imshow(image, cmap='gray', vmin=0, vmax=255, interpolation='nearest')
    axe.set_title(titre)
    axe.axis('off')
plt.tight_layout()
plt.show()
print('Valeurs présentes dans le masque :', np.unique(masque))


**Figure de référence · visible sans exécution**

![Image en gris, masque des pixels lumineux et deux zooms identiques avant et après flou.](../ressources/illustrations/01_gris_flou_masque.png)

En bas, les mêmes détails du bus deviennent moins nets après le flou. En haut à droite, les zones blanches indiquent seulement une intensité supérieure à 150 après flou, pas la présence d’un bus.


Dans les deux zooms, comparez les contours : le flou adoucit les détails,
sans donner de nom à ce qu’il voit. Les zones blanches peuvent appartenir à une façade, un vêtement ou un véhicule.
Ce traitement peut aider une inspection à éclairage contrôlé, mais il n'a pas
appris la notion de bus. **Transformer des pixels et reconnaître une catégorie
sont deux opérations distinctes.**

Nous avons maintenant une raison d'utiliser un modèle appris : les objets varient
en couleur, taille, éclairage et point de vue. Une règle de luminosité seule
ne suffit pas à relier toutes ces apparences au même nom.


## 7. À vous · Garder une zone sans modifier l'original

Complétez `extraire_zone(image, x1, y1, x2, y2)`. La fonction doit renvoyer
une **copie** de `image[y1:y2, x1:x2]`. Les coordonnées reçues sont déjà valides.
Son contrat garde la borne de fin exclue. Le test emploie une petite image
artificielle dont les dimensions sont faciles à vérifier.

Un **test** compare le résultat à ce que nous attendons sur un cas connu.
`assert condition, "message"` signale une erreur si la condition est fausse.
Ici, `None` veut dire « fonction à compléter », pas « zone vide ».


In [ ]:
def extraire_zone(image, x1, y1, x2, y2):
    # À compléter : extraire la zone puis renvoyer une copie indépendante.
    return None


In [ ]:
# Chaque pixel porte des valeurs différentes pour vérifier aussi le contenu extrait.
image_test = np.arange(72, dtype=np.uint8).reshape(4, 6, 3)  # Nombres 0 à 71, rangés en 4 × 6 × 3.
zone_test = extraire_zone(image_test, 1, 1, 4, 3)
if zone_test is None:
    print("Exercice à compléter : la zone doit avoir 2 lignes et 3 colonnes.")
else:
    assert zone_test.shape == (2, 3, 3), "Vérifier l'ordre y puis x et les bornes."
    assert np.array_equal(zone_test, image_test[1:3, 1:4]), "Vérifier les pixels sélectionnés."
    assert not np.shares_memory(zone_test, image_test), "La zone doit être une copie."
    print("Test réussi : dimensions correctes et original indépendant.")


### À vous · Réduire sans déformer

Complétez `reduire_image(image, largeur_voulue)`. Les entrées sont valides et
`largeur_voulue` est positive. Calculez la hauteur proportionnelle avec `round`,
avec un minimum d'un pixel, puis utilisez `cv2.resize` et `INTER_AREA`.
Pourquoi ce minimum ? Une image de hauteur 1 et largeur 100, réduite à une
largeur de 10, donnerait `round(1 × 10 / 100) = 0`. Une image ne peut pas avoir
zéro ligne : on garde au moins une ligne.
Le test attend une image de **100 × 50 pixels** à partir de **200 × 100 pixels**.


In [ ]:
def reduire_image(image, largeur_voulue):
    # À compléter : lire la forme, calculer la hauteur, puis redimensionner.
    return None


In [ ]:
image_test = np.full((100, 200, 3), 180, dtype=np.uint8)  # Image grise connue, pas entièrement nulle.
image_test[:, :100] = 30  # Moitié gauche sombre : vérifier aussi le contenu, pas seulement la taille.
reduction = reduire_image(image_test, 100)
if reduction is None:
    print("Exercice à compléter : conserver les proportions de l'image.")
else:
    assert reduction.shape == (50, 100, 3), "resize attend largeur, hauteur."
    attendue = np.full((50, 100, 3), 180, dtype=np.uint8)
    attendue[:, :50] = 30
    assert np.array_equal(reduction, attendue), "Les deux zones de couleur doivent être conservées."
    assert image_test.shape == (100, 200, 3), "L'original ne doit pas être remplacé."
    aplatie = np.full((1, 100, 3), 180, dtype=np.uint8)
    assert reduire_image(aplatie, 10).shape == (1, 10, 3), "La hauteur minimale vaut un pixel."
    print("Test réussi : largeur divisée par deux, hauteur aussi.")


## 8. Relier les gestes à leur sens

1. Un tableau de forme `(480, 640, 3)` contient combien de lignes, de colonnes et de canaux ?
2. Pourquoi une image lue par OpenCV peut-elle avoir des couleurs étranges dans Matplotlib ?
3. Pourquoi les pixels blancs de notre masque ne sont-ils pas des détections de bus ?
4. Pour compter deux bus dans une photo, faut-il une étiquette globale ou des objets localisés ?


Vos réponses : à compléter.


Nous savons lire une image, retrouver ses pixels et observer ce qu'une transformation
conserve ou retire. Il manque encore le lien entre **apparence** et **catégorie**.
Dans [02 · Classer une image](02_classer_une_image.ipynb), un modèle déjà entraîné
proposera des étiquettes pour notre photo. Nous lirons ses scores avant de lui faire confiance.

Références : [lecture d'images OpenCV](https://docs.opencv.org/4.x/dc/d2e/tutorial_py_image_display.html),
[opérations sur les images](https://docs.opencv.org/4.x/d3/df2/tutorial_py_basic_ops.html),
[seuillage](https://docs.opencv.org/4.x/d7/d4d/tutorial_py_thresholding.html).
Origine des photos et licences : [SOURCES.txt](../SOURCES.txt).
